In [1]:
from langchain_groq import ChatGroq
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import yfinance as yf
from langchain.agents import AgentType, initialize_agent
from langchain_community.tools.yahoo_finance_news import YahooFinanceNewsTool
from langchain_groq import ChatGroq

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

from pymongo.operations import SearchIndexModel
from langchain_community.vectorstores import AtlasDB
# from langchain_mongodb import MongoDBAtlasVectorSearch
# from pymongo import MongoClient
from langchain.chains import RetrievalQA
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.output_parsers import StrOutputParser
import PyPDF2
import pandas as pd
from langchain_core.documents import Document
from typing import Iterable
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter, CharacterTextSplitter, TokenTextSplitter
)
from langchain.text_splitter import NLTKTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables.utils import ConfigurableFieldSpec
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

USER_AGENT environment variable not set, consider setting it to identify your requests.
/home/steeldev/Desktop/Github/PotatoTrade/mlenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-26 12:01:15.572226: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-26 12:01:15.720681: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748241075.788710   24331 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for p

True

In [2]:
"""
Stock Market Analysis Agent using LangGraph and Groq API

This code creates a langgraph agent that:
1. Takes user queries about stocks
2. Fetches real-time stock data from Alpha Vantage API
3. Analyzes the data using Groq's LLM API
4. Returns insights to the user

"""

import os
import json
import requests
from datetime import datetime, timedelta
from typing import Dict, List, Any, Annotated, TypedDict, Sequence
from pydantic import BaseModel, Field

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

import langgraph
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolExecutor
from langgraph.prebuilt.tool_node import ToolNode

# Define API keys (in production, use environment variables)
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "your-groq-api-key")
ALPHAVANTAGE_API_KEY = os.environ.get("ALPHAVANTAGE_API_KEY", "your-alphavantage-api-key")

# Define State schema
class AgentState(TypedDict):
    """Represents the state of our agent."""
    messages: List[Any]
    tools_output: List[Dict[str, Any]]
    next: str

# Initialize the Groq Model
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",  # Using a capable model for financial analysis
    api_key=GROQ_API_KEY,
    temperature=0.1  # Lower temperature for more deterministic responses
)

# Define Stock Analysis Tools



In [16]:

from mongodb_atlas import MongoDBAtlas, MongoDBAtlasDocumentManager
class MongoDBAtlasQA:
    def __init__(self, mongo_uri, db_name, collection_name, embedding, index_name, llm, conversation_id: str= "test_session"):
        """Initialize the MongoDB Atlas QA system."""
        try:
            self.atlas= MongoDBAtlas(db_name, collection_name)
            self.document_manager= MongoDBAtlasDocumentManager(atlas=self.atlas)  
            self.atlas.create_vector_store(
                embedding=embedding,
                index_name=index_name,
                relevance_score_fn="cosine",
            )
            self._prompt = ChatPromptTemplate.from_messages(
                [
                    ("system", "You are a helpful assistant."),
                    # Placeholder for chat history
                    MessagesPlaceholder(variable_name="chat_history"),
                    ("human", "{query}"),
                ]
            )
            self.llm: ChatGroq=llm
            self.llm.bind_tools([self.get_similarity_search])
            self.llm= self._prompt| self.llm | StrOutputParser()
            
            self._config_fields = [
                ConfigurableFieldSpec(
                    id="user_id",
                    annotation=str,
                    name="User ID",
                    description="Unique identifier for a user.",
                    default="",
                    is_shared=True,
                ),
                ConfigurableFieldSpec(
                    id="conversation_id",
                    annotation=str,
                    name="Conversation ID",
                    description="Unique identifier for a conversation.",
                    default="",
                    is_shared=True,
                ),
            ]
            self.llm_with_history=RunnableWithMessageHistory(
                self.llm,
                self.get_chat_history,
                input_messages_key="query",
                history_messages_key="chat_history",
                history_factory_config=self._config_fields
            )
            self.config = {"configurable": {"user_id": "user1", "conversation_id": conversation_id}}
        except Exception as e:
            print(f"Error initializing MongoDBAtlasQA: {e}")

    def get_chat_history(self, user_id, conversation_id):
        try:
            return SQLChatMessageHistory(
                table_name="basic_stock_info_chat_history",
                session_id=conversation_id,
                connection=os.getenv("POSTGRES_URI"),
            )
            
        except Exception as e:  
            print(f"Error retrieving chat history: {e}")
            return None

    @tool
    def get_similarity_search(self, query):
        """Perform a similarity search for info on stock market."""
        try:
            results = self.atlas.similarity_search(query, k=5)
            return results
        except Exception as e:
            print(f"Error performing similarity search: {e}")
            return None

    def query(self, text):
        """Query the knowledge base."""
        try:
            return self.llm_with_history.invoke(
                {"query": text}, self.config
            )
        except Exception as e:
            print(f"Error querying the knowledge base: {e}")
            return None
    
    

In [7]:
MONGO_URI = os.getenv("MONGODB_ATLAS_CLUSTER_URI")
DB_NAME = "potato_trade"
COLLECTION_NAME = "vectorized_documents"
INDEX_NAME = "test_vector_index"

# llm = ChatGroq(model_name="llama3-8b-8192", api_key=os.getenv("GROQ_API_KEY"), async_client=True)
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",  # Using a capable model for financial analysis
    api_key=GROQ_API_KEY,
    temperature=0.1  # Lower temperature for more deterministic responses
)
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [17]:
qa_system = MongoDBAtlasQA(MONGO_URI, DB_NAME, COLLECTION_NAME, embedding, INDEX_NAME, llm)


In [18]:
qa_system.atlas.similarity_search("What is stock market?")

[Document(metadata={'_id': '67e4cdb00e719b3b7e644dfe', 'doc_index': 0, 'chunk_index': 2}, page_content='The share market, also known as the stock market, is a platform where buyers and sellers come together to trade publicly listed shares of companies. The market is regulated by the Securities and Exchange Board of India (SEBI), which oversees the functioning of stock exchanges and ensures that listed companies comply with regulations and disclosure requirements.\n\nIf a company has issued 100 shares and you own 1 share, you own 1% stake in the company. The share market is where shares of different companies are traded.'),
 Document(metadata={'_id': '67e4cdb00e719b3b7e644dfc', 'doc_index': 0, 'chunk_index': 0}, page_content='Investing is a way to grow your money over time by putting it into various assets. One of the popular investment options is stocks. When you invest in stocks, you become a shareholder and can benefit from the company’s profits and growth. However, the stock market 

In [ ]:
@tool
def get_similarity_search(query: str):
    """Perform a similarity search for info on stock market."""
    try:
        qa_system = MongoDBAtlasQA(MONGO_URI, DB_NAME, COLLECTION_NAME, embedding, INDEX_NAME, llm)
        results = qa_system.get_similarity_search(query)
        if results is None:
            print("No results found.")
            return []
        
        return results
    except Exception as e:
        print(f"Error performing similarity search: {e}")
        return None

In [20]:
get_similarity_search.invoke("What is stock market?")

Error performing similarity search: 1 validation error for get_similarity_search
query
  Field required [type=missing, input_value={'self': 'What is stock market?'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/missing
